# fkjee — smallest model that solves JEE / NEET style MCQs

Two tracks, same data, same held-out papers:

| track | model | params |
|---|---|---|
| predict | TinyBERT-L2 cross-encoder | 4.4M |
| predict | MiniLM-L6 / L12 cross-encoder | 22.7M / 33.4M |
| solve | SmolLM2-135M / 360M | 139M / 362M |
| solve | Qwen2.5-0.5B / 1.5B | 494M / 1.5B |

Runtime on 2xT4: rankers ≈ 15–60 min each, generators ≈ 1–4 h each. Kaggle gives ~30 GPU-h/week.

**Before running:** upload this repo to `/kaggle/working/fkjee` (or `git clone` it in the next cell).

In [ ]:
# ===================== fkjee one-shot cell =====================
# clone -> install -> optional HF auth -> data -> ranker sweep -> table
# Kaggle: Settings -> Internet ON.  Optional secret: Add-ons -> Secrets -> HF_TOKEN
import os, pathlib, subprocess, sys

REPO = "https://github.com/Yashhh999/fkjee"
DEST = pathlib.Path("/kaggle/working/fkjee")

if not DEST.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(DEST)], check=True)
os.chdir(DEST)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import HfApi
    print("HF user:", HfApi().whoami()["name"])
except Exception as exc:
    print("No HF_TOKEN secret -> Hub uploads disabled:", type(exc).__name__)

subprocess.run([sys.executable, "-m", "fkjee.data", "prepare", "--data-dir", "data"], check=True)
subprocess.run([sys.executable, "scripts/run_sweep.py", "--only", "tinybert", "minilm",
                "--results", "results"], check=True)
print(pathlib.Path("results/sweep.md").read_text())

# Then, one generator per session (resume with --limit 0? no: rerun the same command):
# !python scripts/run_sweep.py --only qwen2.5-0.5b --results results


In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)
print("cuda:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

In [ ]:
# Point this at wherever the repo lives.
%cd /kaggle/working/fkjee
!pip install -q -e .
!python -m fkjee.data prepare --data-dir data

In [ ]:
import json, pandas as pd

stats = json.load(open("data/stats.json"))
print("unique items:", stats["after_dedupe"], "| leakage:", stats["leakage"])
display(pd.DataFrame([
    {"split": s["split"], "n": s["n"], "papers": s["papers"],
     "modules": s["n_modules"], "rationale": s["rationale_rate"], **s["by_subject"]}
    for s in stats["splits"]
]))

In [ ]:
# Per-module coverage: which syllabus chapters are actually represented.
!head -40 data/coverage.md


In [ ]:
# --- Cheap track first: rankers (predict) ---
# TinyBERT is the floor; MiniLM-L6 is the bet. Takes minutes.
!python scripts/run_sweep.py --only tinybert minilm --results results

In [ ]:
# --- Generative track (solve). Uncomment when you have GPU hours. ---
# !python scripts/run_sweep.py --only smollm2-135m --results results      # ~1-1.5 h
# !python scripts/run_sweep.py --only qwen2.5-0.5b --results results      # ~2-3 h
# !python scripts/run_sweep.py --only qwen2.5-1.5b --results results      # ~3-5 h

In [ ]:
import pathlib
p = pathlib.Path("results/sweep.md")
print(p.read_text() if p.exists() else "no sweep yet")

## How to read the table

- **holdout** — unseen papers (JEE ≥2024, NEET ≥2026). Chance is 0.25.
- **abl_question** — question blanked, options kept. Anything well above 0.25 means the model
  is guessing from option shape, not solving.
- **abl_options** — option text blanked. Should collapse to ~0.25.
- **abl_shuffle** — options permuted. Should track `holdout`; a big drop = letter/position bias.
- **jeebench** — JEE Advanced, out-of-domain. Expect a large drop.

A model only "solves" if `holdout` is far above chance **and** both ablations sit at chance.